# Problem Set 8: Project Genesis – Der souveräne Wächter

Sensible Telemetriedaten, physikalische Anomalien und geschützte Systemzustände wurden über unsichere Netzwerke an externe Server übertragen. Jede Millisekunde Latenz bedrohte die Stabilität eurer Echtzeitschleifen; jeder Netzwerkausfall führte zum sofortigen Zusammenbruch eures digitalen Abbilds.

Heute errichten wir die **Festung der Autarkie**. Ihr werdet ein hochentwickeltes, lokal lauffähiges Modell der **Google Gemma 4**-Familie tief im Fundament eurer eigenen Hardware verankern. Eure Mission ist es, eine ereignisdiskrete Warteschlangensimulation – das pulsierende Herz einer virtuellen Fabrikationsstraße – mit diesem lokalen Wächter zu koppeln. Ohne ein einziges Datenpaket nach außen zu senden, wird der Wächter die Telemetrie überwachen, im geschützten Denkanal logische Analysen durchführen, auf eine lokale Wissensdatenbank zugreifen und regulierend in die Physik der Warteschlange eingreifen.

Bereitet eure Terminals vor. Aktiviert die lokalen Tensor-Kerne. Die Souveränität über die Daten ist der Schlüssel zur absoluten Kontrolle. Let's build!

## Exercise 1: Die Initialisierung des lokalen Wächters (Gemma 4 Edge)

Wir verwenden die hochgradig effiziente Variante **Gemma 4 E2B-IT**. Dieses Modell weist trotz seiner kompakten Größe (2,3 Milliarden effektive Parameter) dank der neuartigen Per-Layer Embeddings (PLE) eine exzellente Logikfähigkeit auf.

### Instruktionen:
1. Melden Sie sich bei [Kaggle](https://www.kaggle.com) an und akzeptieren Sie die Nutzungsbedingungen auf der offiziellen Gemma 4 Modellseite.
2. Erstellen Sie ein API-Token in Ihren Kaggle-Kontoeinstellungen, um die Datei `kaggle.json` herunterzuladen.
3. Setzen Sie die Umgebungsvariablen `KAGGLE_USERNAME` und `KAGGLE_KEY` in Ihrer Python-Laufzeitumgebung.
4. Laden Sie das Modell mit `kagglehub` herunter und initialisieren Sie die Inferenz-Pipeline.

In [9]:
import os
import kagglehub

# Tragen Sie hier Ihre Credentials ein (oder nutzen Sie eine .env Datei)
# os.environ["KAGGLE_USERNAME"] = "ihr_benutzername"
# os.environ["KAGGLE_KEY"] = "ihr_api_schluessel"

try:
    print("Fordere Gewichte von der Kaggle-Schnittstelle an...")
    # Herunterladen der instruction-tuned Gemma 4 E2B Variante über kagglehub
    model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")
    print(f"Modell erfolgreich lokalisiert im Systemcache unter: {model_path}")
except Exception as e:
    print(f"Fehler beim Herunterladen des Modells: {e}")
    print("Sicherstellen, dass die Modell-Lizenz auf Kaggle akzeptiert wurde und die API-Schlüssel korrekt konfiguriert sind.")

c:\Users\nilss\Documents\genesis-oracle\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fordere Gewichte von der Kaggle-Schnittstelle an...
Modell erfolgreich lokalisiert im Systemcache unter: C:\Users\nilss\.cache\kagglehub\models\google\gemma-4\transformers\gemma-4-e2b-it\1


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Initialisiere Inferenz-Pipeline (bfloat16 für VRAM-Effizienz)...")
try:
    processor = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("Lokaler Wächter online und inferenzbereit.")
except NameError:
    print("Modellpfad nicht gefunden. Führen Sie die obige Zelle zuerst aus.")

Initialisiere Inferenz-Pipeline (bfloat16 für VRAM-Effizienz)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 7463.58it/s]


Lokaler Wächter online und inferenzbereit.


**Frage:** Warum ist diese Einsparung (Kompakte Parameter/bfloat16) ein entscheidender Faktor bei der Ausführung paralleler physikalischer Simulationen auf derselben Hardware?

Die reduzierte Speicherlast ist entscheidend, weil das lokale Sprachmodell und die physikalische Simulation gleichzeitig um denselben RAM beziehungsweise VRAM konkurrieren. Wenn das Modell durch kompakte Parameter und bfloat16 weniger Speicher benötigt, bleibt mehr Speicherbandbreite und Rechenkapazität für parallele Simulationspfade, Warteschlangenmodelle und Telemetriepuffer verfügbar. Dadurch sinkt die Gefahr, dass das System durch Speicherauslagerung, GPU-Überlauf oder starke Latenzspitzen instabil wird. Gerade bei Edge-Systemen ist das wichtig, weil dort keine unbegrenzten Cloud-Ressourcen zur Verfügung stehen und die Regelung trotzdem lokal und echtzeitnah reagieren muss.

## Exercise 2: Aktivierung des Cognitive Core (Der Denkmodus)

Dieser Prozess wird nativ gestartet, indem das Token `<|think|>` am Anfang der Systemnachricht platziert wird. Das Modell gibt seine logischen Gedankenschritte innerhalb der Trenner `<|channel>thought\n` und `<channel|>` aus.

### Instruktionen:
1. Implementieren Sie eine Inferenz-Funktion `generate_thoughtful_response`, die den System-Prompt so formatiert, dass der native Denkmodus aktiviert wird.
2. Nutzen Sie reguläre Ausdrücke (RegEx), um den Denkanal (`thought`) von der finalen Antwort zu trennen.
3. Schreiben Sie eine Hilfsfunktion, die die Gedankenblöcke aus der Historie entfernt, bevor die nächste Benutzerinteraktion stattfindet.

In [11]:
import re
import torch


def extract_thought_and_final_response(raw_output: str) -> tuple[str, str]:
    """
    Extrahiert den Thought-Channel und die finale Antwort aus der Modellausgabe.
    """

    if raw_output is None:
        raw_output = ""

    # Fall 1: vollständiger Thought-Block mit Start- und Endmarker
    thought_pattern = r"<\|channel\>thought\s*(.*?)\s*<channel\|>"

    thought_match = re.search(
        thought_pattern,
        raw_output,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if thought_match:
        thought_content = thought_match.group(1).strip()

        clean_response = re.sub(
            thought_pattern,
            "",
            raw_output,
            flags=re.DOTALL | re.IGNORECASE,
        ).strip()

    # Fall 2: Thought-Block beginnt, aber Endmarker fehlt wegen Tokenlimit
    elif "<|channel>thought" in raw_output:
        thought_content = raw_output.split("<|channel>thought", 1)[1].strip()

        clean_response = (
            "Keine separate finale Antwort gefunden, "
            "weil das Modell innerhalb des Tokenlimits im Thought-Channel blieb."
        )

    # Fall 3: Modell gibt keinen Thought-Block aus
    else:
        thought_content = "Kein expliziter Denkprozess aufgezeichnet."
        clean_response = raw_output.strip()

    clean_response = (
        clean_response
        .replace("<turn|>", "")
        .replace("<bos>", "")
        .replace("<eos>", "")
        .strip()
    )

    return thought_content, clean_response


def generate_thoughtful_response(system_prompt, user_message, max_new_tokens=512):
    messages = [
        {"role": "system", "content": f"<|think|>\n{system_prompt}"},
        {"role": "user", "content": user_message}
    ]
    
    # Erstellung des Chat-Templates über den integrierten Prozessor
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Konvertierung in Eingabe-Tensors
    inputs = processor(text=prompt_text, return_tensors="pt").to(model.device)
    
    # Inferenz ohne Gradientenberechnung zur Performance-Optimierung
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.eos_token_id,
            eos_token_id=processor.eos_token_id
        )
    
    # Isolation der generierten Tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded_output = processor.decode(generated_tokens, skip_special_tokens=False)
    
    # RegEx-Auswertung über Hilfsfunktion
    thought_content, clean_response = extract_thought_and_final_response(decoded_output)
    
    return thought_content, clean_response

In [12]:
def clean_history(messages):
    """
    Hilfsfunktion: Entfernt Gedankenblöcke aus der Historie (State-Management).
    Dies verhindert eine kognitive Überladung und Token-Wachstum bei Multi-Turn Chats.
    """

    cleaned_messages = []
    thought_pattern = r"<\|channel\>thought\s*.*?\s*<channel\|>"

    for msg in messages:
        if msg["role"] == "model":
            clean_content = re.sub(
                thought_pattern,
                "",
                msg["content"],
                flags=re.DOTALL | re.IGNORECASE,
            ).strip()

            # Falls ein Thought-Block angefangen, aber nicht sauber beendet wurde
            if "<|channel>thought" in clean_content:
                clean_content = clean_content.split("<|channel>thought", 1)[0].strip()

            clean_content = (
                clean_content
                .replace("<turn|>", "")
                .replace("<bos>", "")
                .replace("<eos>", "")
                .strip()
            )

            cleaned_messages.append(
                {
                    "role": "model",
                    "content": clean_content,
                }
            )
        else:
            cleaned_messages.append(msg)

    return cleaned_messages

In [13]:
test_system_prompt = """
You are the local Sovereign Sentinel.
Analyze factory telemetry and answer as an industrial controller.
"""

test_user_message = """
The queue length is rising quickly while the service rate is low.
What should the controller do?
"""

thought, final_response = generate_thoughtful_response(
    test_system_prompt,
    test_user_message,
    max_new_tokens=512
)

print("=== EXTRACTED THOUGHT CHANNEL ===")
print(thought)

print("\n=== FINAL RESPONSE ===")
print(final_response)

=== EXTRACTED THOUGHT CHANNEL ===
Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request & Persona:**
    *   **Role:** Local Sovereign Sentinel / Industrial Controller. (This requires a technical, decisive, and systematic approach, focused on optimization and stability.)
    *   **Input Data (The Problem):** "The queue length is rising quickly while the service rate is low."
    *   **Goal:** Determine the corrective actions (What should the controller do?).

2.  **Deconstruct the Problem (Identify the Root Cause Categories):**
    *   *Queue Length Rising Quickly* $\rightarrow$ Demand > Capacity (or bottleneck).
    *   *Service Rate is Low* $\rightarrow$ Processing speed is insufficient.
    *   *Combined Implication:* The system is overloaded, or the bottleneck is limiting throughput.

3.  **Determine the Controller's Action Strategy (The 4 Pillars of Control):**
    *   **Pillar 1: Immediate Mitigation (Stop the Bleeding).**
    *   **Pillar 2: 

In [14]:
dirty_history = [
    {
        "role": "user",
        "content": "Analyze the factory queue.",
    },
    {
        "role": "model",
        "content": """<|channel>thought
The model privately reasons about overload.
<channel|>
The queue is unstable and service rate should increase.""",
    },
]

cleaned = clean_history(dirty_history)

print("=== CLEANED HISTORY ===")
for message in cleaned:
    print(message)

=== CLEANED HISTORY ===
{'role': 'user', 'content': 'Analyze the factory queue.'}
{'role': 'model', 'content': 'The queue is unstable and service rate should increase.'}


## Exercise 3: Die diskrete Warteschlangensimulation (Foundry)

Teile kommen an, warten im Puffer und werden von einer Maschine abgearbeitet. Wir simulieren dieses Verhalten stochastisch. Die Ankunftszeiten folgen einem Poisson-Prozess mit Rate $\lambda$ (Arrival Rate), die Bearbeitungszeiten an der Station folgen einer Exponential- bzw. Poisson-Verteilung mit Rate $\mu$ (Service Rate).

### Instruktionen:
1. Schreiben Sie eine Simulationsklasse `DiscreteFoundry`, die den Zustand eines Puffers abbildet.
2. Wenn die Ankunftsrate die Bearbeitungsrate übersteigt (Instabilität), füllt sich der Puffer.
3. Bei Erreichen der maximalen Kapazität kommt es zum Pufferüberlauf (Datenverlust/Ausschuss).
4. Die Methode `step()` muss die Systemvariablen aktualisieren und einen JSON-String exportieren.

In [15]:
import numpy as np
import json

class DiscreteFoundry:
    def __init__(self, arrival_rate=2.5, service_rate=1.0, capacity=15):
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate
        self.capacity = capacity
        self.buffer = 0
        self.dropped = 0
        self.processed = 0
        
    def step(self):
        # Stochastische Approximation
        arrivals = np.random.poisson(self.arrival_rate)
        services = np.random.poisson(self.service_rate)
        
        self.buffer += arrivals
        
        # Überlauf prüfen
        if self.buffer > self.capacity:
            overflow = self.buffer - self.capacity
            self.dropped += overflow
            self.buffer = self.capacity
            
        # Verarbeitung
        processed_now = min(self.buffer, services)
        self.buffer -= processed_now
        self.processed += processed_now
        
        metrics = {
            "queue_length": int(self.buffer),
            "max_capacity": int(self.capacity),
            "dropped_parts": int(self.dropped),
            "processed_parts": int(self.processed),
            "current_service_rate": float(self.service_rate),
            "arrival_rate": float(self.arrival_rate),
            "unstable": self.arrival_rate > self.service_rate
        }

        return json.dumps(metrics)

In [16]:
# Instanziierung der Produktionslinie
production_line = DiscreteFoundry()
print("Initialer Tick:", production_line.step())

Initialer Tick: {"queue_length": 1, "max_capacity": 15, "dropped_parts": 0, "processed_parts": 2, "current_service_rate": 1.0, "arrival_rate": 2.5, "unstable": true}


## Exercise 4: Geschlossener Regelkreis mit kognitiver Instanz

Die Telemetriedaten werden an den lokalen Wächter übergeben. Dieser analysiert die Situation offline im Denkanal, trifft eine kognitive Entscheidung und gibt einen neuen Wert für die Service-Rate vor, welcher unmittelbar in die Simulation zurückgeführt wird.

### Instruktionen:
1. Konzipieren Sie einen System-Prompt für Gemma 4, der das Modell anweist, als industrieller Regler zu agieren.
2. Das Modell muss den Systemzustand bewerten und die Service-Rate anpassen (Intervall: $[0.5, 3.0]$).
3. Enforcieren Sie ein klares Ausgabe-Format, um den neuen Parameter numerisch parsen zu können.
4. Lassen Sie die gesteuerte Simulation über 8 Zeitschritte laufen.

In [17]:
import json
import re
import torch


sentinel_system_prompt = """
<|think|>
Du bist ein lokaler industrieller KI-Regler für eine diskrete Warteschlange.

Analysiere kurz die Telemetrie.
Danach musst du zwingend eine einzelne maschinenlesbare Steuerzeile ausgeben.

Regeln:
- service_rate muss im Bereich [0.5, 3.0] bleiben.
- Wenn unstable true ist, erhöhe die service_rate.
- Wenn queue_length > 8 oder dropped_parts > 0, setze service_rate deutlich höher.
- Wenn queue_length niedrig ist, aber unstable true ist, erhöhe trotzdem leicht.
- Antworte am Ende zwingend exakt im Format:

NEW_SERVICE_RATE: <zahl>

Keine zusätzlichen Wörter nach dieser Zeile.
"""


def ask_sentinel_for_service_rate(telemetry_json: str, current_service_rate: float):
    """
    Übergibt Telemetrie an das lokale Modell und extrahiert eine neue Service-Rate.
    """

    user_message = f"""
Aktuelle Telemetrie:
{telemetry_json}

Aktuelle Service-Rate:
{current_service_rate}

Entscheide die neue Service-Rate.
Antworte am Ende zwingend im Format:
NEW_SERVICE_RATE: <wert>
"""

    messages = [
        {
            "role": "system",
            "content": sentinel_system_prompt,
        },
        {
            "role": "user",
            "content": user_message,
        },
    ]

    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=prompt_text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=processor.eos_token_id,
            eos_token_id=processor.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_output = processor.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

    thought, final_response = extract_thought_and_final_response(raw_output)

    # Wichtig: Wir suchen im gesamten Output, nicht nur in der final_response.
    # Manche lokalen Modelle bleiben lange im Thought-Channel.
    match = re.search(
        r"NEW_SERVICE_RATE:\s*([0-9]+(?:\.[0-9]+)?)",
        raw_output,
        flags=re.IGNORECASE,
    )

    if match:
        parsed_rate = float(match.group(1))
        parsed_rate = max(0.5, min(3.0, parsed_rate))
        parsing_successful = True
    else:
        # Sicherheits-Fallback, falls das Modell keine parsebare Steuerzeile erzeugt.
        telemetry = json.loads(telemetry_json)

        queue_length = telemetry["queue_length"]
        dropped_parts = telemetry["dropped_parts"]
        unstable = telemetry["unstable"]

        if dropped_parts > 0:
            parsed_rate = 3.0
        elif queue_length > 10:
            parsed_rate = 3.0
        elif queue_length > 5:
            parsed_rate = min(3.0, current_service_rate + 0.7)
        elif unstable:
            parsed_rate = min(3.0, current_service_rate + 0.5)
        else:
            parsed_rate = current_service_rate

        parsing_successful = False

    return {
        "raw_output": raw_output,
        "thought": thought,
        "final_response": final_response,
        "new_service_rate": parsed_rate,
        "parsing_successful": parsing_successful,
    }


production_line = DiscreteFoundry()

print("=== START DES GESCHLOSSENEN REGELKREISES ===")
print(f"Initiale Service-Rate: {production_line.service_rate}")
print()

for tick in range(8):
    telemetry_json = production_line.step()
    telemetry_data = json.loads(telemetry_json)

    print("=" * 80)
    print(f"[Tick {tick + 1}] Telemetrie-Eingang:")
    print(telemetry_json)

    sentinel_result = ask_sentinel_for_service_rate(
        telemetry_json=telemetry_json,
        current_service_rate=production_line.service_rate,
    )

    print("\n[Denkanal gekürzt]:")
    print(sentinel_result["thought"][:500])

    print("\n[Finale Modellantwort]:")
    print(sentinel_result["final_response"])

    print("\n[Parsing]:")
    print("Parsing erfolgreich:", sentinel_result["parsing_successful"])
    print("Neue Service-Rate:", sentinel_result["new_service_rate"])

    production_line.service_rate = sentinel_result["new_service_rate"]

    print("\n[System-Aktion]:")
    print(f"Service-Rate für den nächsten Schritt gesetzt auf {production_line.service_rate}")
    print("\n=== FINALER REGELKREIS-STATUS ===")
    print(f"Letzte Service-Rate: {production_line.service_rate}")
    print(f"Aktueller Buffer: {production_line.buffer}")
    print(f"Verarbeitete Teile gesamt: {production_line.processed}")
    print(f"Verworfene Teile gesamt: {production_line.dropped}")

=== START DES GESCHLOSSENEN REGELKREISES ===
Initiale Service-Rate: 1.0

[Tick 1] Telemetrie-Eingang:
{"queue_length": 1, "max_capacity": 15, "dropped_parts": 0, "processed_parts": 0, "current_service_rate": 1.0, "arrival_rate": 2.5, "unstable": true}

[Denkanal gekürzt]:
Thinking Process:

1.  **Analyze the input telemetry:**
    *   `queue_length`: 1
    *   `max_capacity`: 15
    *   `dropped_parts`: 0
    *   `processed_parts`: 0
    *   `current_service_rate`: 1.0
    *   `arrival_rate`: 2.5
    *   `unstable`: true

2.  **Analyze the rules for determining `service_rate`:**
    *   **Base constraint:** `service_rate` must be in the range [0.5, 3.0].
    *   **Rule 1:** If `unstable` is true, increase

[Finale Modellantwort]:
Keine separate finale Antwort gefunden, weil das Modell innerhalb des Tokenlimits im Thought-Channel blieb.

[Parsing]:
Parsing erfolgreich: False
Neue Service-Rate: 1.5

[System-Aktion]:
Service-Rate für den nächsten Schritt gesetzt auf 1.5

=== FINALER REGEL

## Exercise 5: Cloud-gestütztes Fine-Tuning über die Colab CLI

Um das daten- und rechenintensive Modell-Training auszulagern, ohne den lokalen Terminal-Workflow zu unterbrechen, nutzen wir die neue **Google Colab CLI** (`google-colab-cli`). Sie ermöglicht es uns, über ein einziges Kommando Remote-GPUs im Google-Rechenzentrum zu provisionieren, Skripte auszuführen und Trainings-Ergebnisse (LoRA-Adapter) direkt in unseren lokalen Workspace herunterzuladen.

### Szenario:
Wir wollen ein feingetuntes **Gemma 4 E2B**-Modell über QLoRA trainieren, damit es Telemetrieberichte direkt in formstabilen industriellen Steuerungscodes formatiert.

### Instruktionen:
1. Installieren Sie die offizielle Google Colab CLI in Ihrer Terminal-Umgebung.
2. Erstellen Sie ein lokales Python-Skript `finetune_run.py`, das Gemma 4 unter Verwendung von KerasNLP/Transformers für ein LoRA-Feintuning initialisiert.
3. Verwenden Sie die Colab CLI-Befehlssequenz, um ein Remote-System mit einer T4 GPU anzufordern, die Pakete zu installieren, das Skript auszuführen und die fertig trainierten Gewichts-Adapter (`safetensors`) zurück in Ihren lokalen Ordner zu übertragen.

In [18]:
from pathlib import Path

script_path = Path("../src/finetune_run.py")

print("Aktueller Notebook-Arbeitsordner:", Path.cwd())
print("Skriptpfad:", script_path)
print("finetune_run.py vorhanden:", script_path.exists())

if script_path.exists():
    print("\n=== Vorschau auf finetune_run.py ===")
    print(script_path.read_text(encoding="utf-8")[:1200])

Aktueller Notebook-Arbeitsordner: c:\Users\nilss\Documents\genesis-oracle\Notebooks
Skriptpfad: ..\src\finetune_run.py
finetune_run.py vorhanden: True

=== Vorschau auf finetune_run.py ===
import keras
import keras_hub
import os

os.environ["KERAS_BACKEND"] = "jax"

print("Lade Gemma 4 E2B für LoRA Fine-Tuning...")
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")

# 2. Aktivierung von Low-Rank Adaptation (LoRA)
gemma_lm.backbone.enable_lora(rank=4)

# 3. Dummy-Trainingsdaten formatieren
training_data = [
    {"prompt": "Status: queue_length=14. Action?", "response": "NEW_SERVICE_RATE: 3.0"},
    {"prompt": "Status: queue_length=1. Action?", "response": "NEW_SERVICE_RATE: 0.5"}
]
prompts = [f"{item['prompt']} -> {item['response']}" for item in training_data]

# 4. Kompilierung des Modells mit optimierter Lernrate
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    weight

In [19]:
# Optionale Installation der CLI im lokalen Terminal:
# pip install google-colab-cli

In [20]:
finetune_script = """
import keras
import keras_hub
import os

os.environ["KERAS_BACKEND"] = "jax"

print("Lade Gemma 4 E2B für LoRA Fine-Tuning...")
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")

# 2. Aktivierung von Low-Rank Adaptation (LoRA)
gemma_lm.backbone.enable_lora(rank=4)

# 3. Dummy-Trainingsdaten formatieren
training_data = [
    {"prompt": "Status: queue_length=14. Action?", "response": "NEW_SERVICE_RATE: 3.0"},
    {"prompt": "Status: queue_length=1. Action?", "response": "NEW_SERVICE_RATE: 0.5"}
]
prompts = [f"{item['prompt']} -> {item['response']}" for item in training_data]

# 4. Kompilierung des Modells mit optimierter Lernrate
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    weighted_metrics=["accuracy"]
)

# 5. Training für eine repräsentative Epoche
gemma_lm.fit(prompts, epochs=1, batch_size=2)

# 6. Sichern des trainierten Adapters im VM-Speicher
gemma_lm.save_to_preset("./gemma4_lora_adapter")
print("Feintuning erfolgreich abgeschlossen. Gewichte lokal in der Colab-VM gespeichert.")
"""

from pathlib import Path

script_path = Path("../src/finetune_run.py")
script_path.parent.mkdir(parents=True, exist_ok=True)

with open(script_path, "w", encoding="utf-8") as f:
    f.write(finetune_script.strip())

print(f"Lokales Skript erstellt und bereit zum Versand: {script_path}")
print("Absoluter Pfad:", script_path.resolve())

Lokales Skript erstellt und bereit zum Versand: ..\src\finetune_run.py
Absoluter Pfad: C:\Users\nilss\Documents\genesis-oracle\src\finetune_run.py


Das folgende Kommando veranschaulicht den transparenten Code-Uplink in der Bash-Shell:

```bash
# 1. Allokieren Sie eine neue Remote-VM mit einer performanten T4-GPU
colab new -s gemma-tuning --gpu T4

# 2. Installieren Sie die modernsten ML-Bibliotheken direkt auf der Remote-Instanz via 'uv'
colab install -s gemma-tuning keras-hub keras tensorflow torch

# 3. Führen Sie Ihr lokales Python-Skript direkt auf der remote GPU aus (Transparenter Code-Uplink)
colab exec -s gemma-tuning -f finetune_run.py

# 4. Laden Sie das Ergebnis (den fertigen LoRA-Gewichts-Adapter) sicher herunter
colab download -s gemma-tuning /content/gemma4_lora_adapter ./local_gemma4_adapter

# 5. Beenden Sie die Remote-Sitzung, um Ihre Rechenressourcen zu schonen
colab stop -s gemma-tuning
```

**Frage:** Wie schützt das Zusammenspiel aus Colab CLI und lokalem Edge-Modell die geistige Souveränität Ihrer Simulationsdaten?

### Reflexion: Schutz der geistigen Souveränität

Das Zusammenspiel aus Colab CLI und lokalem Edge-Modell schützt die geistige Souveränität der Simulationsdaten, weil die eigentliche Produktionssimulation und ihre sensiblen Telemetriedaten lokal bleiben können. Die Cloud wird nur gezielt für rechenintensive Trainingsaufgaben genutzt, zum Beispiel um LoRA-Adapter auf einer temporären T4-GPU zu trainieren. Dadurch muss nicht die gesamte operative Simulationsumgebung dauerhaft an einen externen Dienst übertragen werden. Nach dem Remote-Training werden nur die erzeugten Adaptergewichte wieder in den lokalen Workspace heruntergeladen und dort mit dem lokalen Sentinel-Modell kombiniert. So bleibt die Kontrolle über Systemlogik, Telemetrie und produktive Entscheidungsprozesse beim lokalen Betreiber.

## Exercise 6: Kontext-Injektion (Lokales RAG für SOPs)

Der Wächter muss spezifische Maschinenvorgaben (Standard Operating Procedures, SOPs) heranziehen, um den optimalen Modus auszuwählen. In dieser Übung implementieren Sie ein offline-fähiges RAG-System, das auf Basis der aktuellen Queue-Länge die passende Regelungsvorschrift aus einem lokalen Dokumentenindex extrahiert und als Kontext in das Gemma 4-Modell einspeist.

### Instruktionen:
1. Erstellen Sie eine Liste von SOPs als lokalen Text-Index.
2. Implementieren Sie eine einfache, abstandsbasierte Suchfunktion (z. B. Keyword-Matching oder Cosine-Similarity über Wortfrequenzen), um das relevanteste SOP-Dokument basierend auf der Queue-Länge zu filtern.
3. Integrieren Sie dieses Dokument als zusätzlichen Kontext in den Prompt für Gemma 4.

In [21]:
# Lokaler RAG-Index
SOP_DATABASE = {
    "SOP-101_NORMAL": "Warteschlange < 5. Maschine im Eco-Modus belassen. Rate auf 1.0 drosseln.",
    "SOP-202_WARNING": "Warteschlange >= 5 und <= 10. Maschine in Normalbetrieb. Rate auf 2.0 erhöhen.",
    "SOP-999_CRITICAL": "Warteschlange > 10. NOTFALL-PROTOKOLL. Rate zwingend auf 3.0 setzen!"
}

def retrieve_relevant_sop(queue_length):
    if queue_length > 10:
        return SOP_DATABASE["SOP-999_CRITICAL"]
    elif queue_length >= 5:
        return SOP_DATABASE["SOP-202_WARNING"]
    else:
        return SOP_DATABASE["SOP-101_NORMAL"]

# Testlauf des lokalen RAG-Systems
test_queue = 12
retrieved_sop = retrieve_relevant_sop(test_queue)
print(f"Extrahierte SOP für Warteschlangenlänge {test_queue}:\n-> {retrieved_sop}")

Extrahierte SOP für Warteschlangenlänge 12:
-> Warteschlange > 10. NOTFALL-PROTOKOLL. Rate zwingend auf 3.0 setzen!


In [22]:
# Demonstration der RAG-gestützten Prompt-Injektion

base_sentinel_prompt = """
Du bist der lokale Sovereign Sentinel einer industriellen Produktionslinie.
Analysiere die aktuelle Warteschlangen-Telemetrie und wähle eine passende Service-Rate.
Antworte am Ende im Format:
NEW_SERVICE_RATE: <wert>
"""

augmented_prompt = f"{base_sentinel_prompt}\n\nWICHTIGE VORSCHRIFT (SOP) FÜR DIESEN SCHRITT:\n{retrieved_sop}"
demo_query = f"Statusbericht: Queue={test_queue}, Capacity=15. Aktuelle Service-Rate: 1.0. Entscheidung?"

try:
    t, a = generate_thoughtful_response(augmented_prompt, demo_query, max_new_tokens=256)
    print("\n=== RAG-AUGMENTIERTE ENTSCHEIDUNG ===")
    print("Verwendete SOP:\n", retrieved_sop)
    print("\nGedanken:\n", t)
    print("\nAktion:\n", a)
except NameError as error:
    print("Bitte stellen Sie sicher, dass das Gemma-Modell und generate_thoughtful_response geladen wurden.")
    print("Fehler:", error)


=== RAG-AUGMENTIERTE ENTSCHEIDUNG ===
Verwendete SOP:
 Warteschlange > 10. NOTFALL-PROTOKOLL. Rate zwingend auf 3.0 setzen!

Gedanken:
 Thinking Process:

1.  **Analyze the Request:** The user is asking me, acting as the Sovereign Sentinel of an industrial production line, to analyze the queue telemetry and choose an appropriate service rate.
    *   Current Status: Queue = 12, Capacity = 15.
    *   Current Service Rate: 1.0.
    *   Goal: Determine the new Service Rate.

2.  **Consult the SOP (Standard Operating Procedure):**
    *   Rule: Warteschlange > 10. NOTFALL-PROTOKOLL. Rate zwingend auf 3.0 setzen! (Queue > 10. EMERGENCY PROTOCOL. Rate must be set to 3.0!)

3.  **Apply the SOP to the Data:**
    *   Queue is 12.
    *   12 > 10.
    *   Therefore, the Emergency Protocol must be activated.

4.  **Determine the Required Service Rate:**
    *   According to the SOP, the rate must be 3.0.

5.  **Format the Output:** The required output format is `NEW_SERVICE_RATE: <wert>`.

Akt

**Frage zur Reflexion:**
Inwiefern erleichtert dieser Ansatz die Anpassung der Systemlogik, wenn sich die Werksvorschriften ändern?

Der lokale RAG-Ansatz erleichtert die Anpassung der Systemlogik, weil Werksvorschriften nicht fest im Modell oder tief im Simulationscode eingebaut werden müssen. Stattdessen liegen die Standard Operating Procedures in einem lokalen Index und können dort gezielt aktualisiert werden, wenn sich Grenzwerte oder Handlungsanweisungen ändern. Der Sentinel ruft zur Laufzeit abhängig von der aktuellen Queue-Länge die passende SOP ab und nutzt sie als zusätzlichen Kontext für seine Entscheidung. Dadurch bleibt die Steuerlogik wartbar, nachvollziehbar und offline nutzbar, ohne dass sensible Produktionsdaten an externe Dienste übertragen werden müssen.